Här ska vi testa att krympa ett dataset (car_price_dataset.csv) med PCA (Principal Component Analysis) innan vi tränar modellen, och se hur det påverkar resultatet. PCA slår ihop kolumner som liknar varandra för att göra datasetet mindre, men nackdelen är att vi tappar lite information.

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error

# 1. Läs in datan (tvingar Pandas att dela upp vid semikolon)
df = pd.read_csv("car_price_dataset.csv", sep=';')

# 2. Preprocessing
# Gör om textkolumner (t.ex. bränsletyp, märke) till dummy-variabler
df = pd.get_dummies(df, drop_first=True, dtype=int)

# Vi vill förutsäga priset (med stort P)
X = df.drop(columns=['Price'])
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Skalning är jätteviktigt innan man gör PCA!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- STEG A: Modell UTAN PCA (för att ha något att jämföra med) ---
model_no_pca = LinearRegression()
model_no_pca.fit(X_train_scaled, y_train)
y_pred_no_pca = model_no_pca.predict(X_test_scaled)
rmse_no_pca = np.sqrt(mean_squared_error(y_test, y_pred_no_pca))

# --- STEG B: Modell MED PCA ---
# Vi ber PCA behålla 95% av all viktig information (varians)
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

model_pca = LinearRegression()
model_pca.fit(X_train_pca, y_train)
y_pred_pca = model_pca.predict(X_test_pca)
rmse_pca = np.sqrt(mean_squared_error(y_test, y_pred_pca))

# 4. Jämför resultaten
print(f"Antal dimensioner (kolumner) utan PCA: {X_train_scaled.shape[1]}")
print(f"Antal dimensioner efter PCA: {pca.n_components_}")
print("-" * 30)
print(f"RMSE utan PCA: {rmse_no_pca:.2f}")
print(f"RMSE med PCA: {rmse_pca:.2f}")

Antal dimensioner (kolumner) utan PCA: 48
Antal dimensioner efter PCA: 35
------------------------------
RMSE utan PCA: 64.91
RMSE med PCA: 870.75
